# Where should we look? Field verification planner + supply impact

Every field has a model probability of silage, so the county estimate has a range. This notebook finds the **few fields whose check by an enumerator shrinks that range the most**, shows what an enumerator's reports do to the estimate, and turns silage acres into **grain bushels not produced**.

Uses real per-field results from `results/` when they exist, otherwise a clearly marked demo county. Add `QUICKSTATS_KEY=...` to `.env` at the repo root to use the live Maryland grain yield.

In [ ]:
# 1. Setup
import os, sys
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline tests (no credits)
from fieldvision_agents.planner import test_planner as t
for name, fn in vars(t).items():
    if name.startswith('test_'):
        fn(); print('PASS', name)

In [ ]:
# 3. Load per-field results (real if available, else demo)
from fieldvision_agents.planner.planner import load_fields, demo_fields, county_status
real = sorted(Path('results').glob('*_*_*.parquet')) + sorted(Path('results').glob('*_*_*.csv'))
fields = load_fields(real[-1]) if real else demo_fields()
print('source:', real[-1] if real else 'DEMO county (placeholder data)')
status = county_status(fields)
status

In [ ]:
# 4. Best 16 visits (2 enumerators x 8 fields a day)
import pandas as pd
from fieldvision_agents.planner.planner import plan_visits
plan = plan_visits(fields, k=16)
print(f"90% range: ±{plan['plus_minus_before']:,} → ±{plan['plus_minus_after']:,} acres "
      f"({plan['range_reduction_pct']}% narrower) by visiting {plan['acres_visited']:,} acres")
pd.DataFrame(plan['fields'])

In [ ]:
# 5. How much each extra visit buys
import matplotlib.pyplot as plt
from fieldvision_agents.planner.planner import uncertainty_curve
pts = pd.DataFrame(uncertainty_curve(fields, max_visits=80)['points'])
ax = pts.plot(x='visits', y='plus_minus', marker='o', legend=False, figsize=(7, 3.5))
ax.set(xlabel='Field visits (best order)', ylabel='± silage acres (90% range)', title='Uncertainty vs field visits')
ax.grid(alpha=0.3); Path('figures').mkdir(exist_ok=True); plt.savefig('figures/planner_curve.png', dpi=150, bbox_inches='tight')

In [ ]:
# 6. Simulate the enumerators' reports and update the estimate (SIMULATED: drawn from the model's own probabilities)
import numpy as np
from fieldvision_agents.planner.planner import apply_reports
rng = np.random.default_rng(0)
reports = {f['field_id']: ('silage' if rng.random() < (f['p_silage'] if f['p_silage'] is not None else status['prior_for_abstained']) else 'grain') for f in plan['fields']}
after = county_status(apply_reports(fields, reports))
print('before:', status['silage_acres'], '±', status['plus_minus'], '| after reports:', after['silage_acres'], '±', after['plus_minus'])

In [ ]:
# 7. Supply impact (live Maryland yield if QUICKSTATS_KEY is set, else the U.S. 2025 yield)
from fieldvision_agents.planner.planner import fetch_state_grain_yield, supply_impact
gy, src = fetch_state_grain_yield('MD', 2022)
impact = supply_impact(status, grain_yield=(gy, src) if gy else None)
impact

In [ ]:
# 8. Ask the planner agent (Nemotron Lightning)
from fieldvision_agents.planner.planner import ask_planner
for q in ['We have 2 enumerators for one day, about 8 visits each. Where should they go and what do we gain?',
          'No budget yet: how many visits are worth it?',
          'What does this silage estimate mean for grain supply?']:
    a = ask_planner(fields, q, grain_yield=(gy, src) if gy else None)
    print('Q:', q, '\nA:', a['answer'], '\n   tools=', a['tools_used'], 'grounded=', a['grounded'], '\n')